### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="miami_housing",
    dataset_year="2016", # data from 2016, curated in 2021
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    # We were unable to find the true original source from the paper, but this is the oldest available data source.
    original_dataset_source_download_link="https://www.openml.org/d/43093",
    download_description="""
wget https://www.openml.org/data/download/22047757/miami2016.arff && \
mkdir -p local-data-warehouse/miami_housing && \
mv miami2016.arff local-data-warehouse/miami_housing/
""",
    # References
    academic_reference_bibtex="""@article{bourassa2021big,
  title={Big data, accessibility and urban house prices},
  author={Bourassa, Steven C and Hoesli, Martin and Merlin, Louis and Renne, John},
  journal={Urban Studies},
  volume={58},
  number={15},
  pages={3176--3195},
  year={2021},
  publisher={SAGE Publications Sage UK: London, England}
}
""",
    academic_reference_bibtex_key="bourassa2021big",
    license="CC BY-NC-SA",
    data_tags=["IID", "Spatial"],
    curation_comments="""
- We log scale the target.
- We drop duplicated homes (non-unique identifiers and location) (~1% of the data).
- We drop the ID column "PARCELNO".
- Anomaly: while the original paper describes data with 57k samples, we only have 13k on OpenML or Kaggle.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="SALE_PRC",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd
import arff
import numpy as np

with open(f"{dataset_mold.path}/miami2016.arff") as f:
    data = arff.load(f)

df = pd.DataFrame(data["data"], columns=[x[0] for x in data["attributes"]])
df = df.drop_duplicates(subset=["PARCELNO"])
df = df.drop(columns=["PARCELNO"])

cat_features = [
    "avno60plus"
]
df[cat_features] = df[cat_features].astype("category")
df[task_mold.target_column_name] = np.log(df[task_mold.target_column_name])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 13,776
Columns: 16
Use sampling: False (sample size: 13,776)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['LATITUDE', 'LONGITUDE', 'CNTR_DIST', 'SUBCNTR_DI', 'OCEAN_DIST', 'RAIL_DIST', 'WATER_DIST', 'HWY_DIST', 'SPEC_FEAT_VAL', 'LND_SQFOOT']
Rows remaining as candidates after top-10 filter: 0 (of 13,776)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,LATITUDE,LONGITUDE,SALE_PRC,LND_SQFOOT,TOT_LVG_AREA,SPEC_FEAT_VAL,RAIL_DIST,OCEAN_DIST,WATER_DIST,CNTR_DIST,SUBCNTR_DI,HWY_DIST,age,avno60plus,month_sold,structure_quality
0,25.953531,-80.343141,12.594731,4500.0,1583.0,2182.0,25820.3,73552.7,1304.1,81077.3,79137.1,426.6,26.0,0.0,9.0,5.0
1,25.532208,-80.379834,12.690349,5000.0,2120.0,1540.0,13638.4,15942.3,10847.7,108098.0,60813.3,497.4,0.0,0.0,9.0,4.0
2,25.971635,-80.181985,12.100712,7500.0,1063.0,0.0,5077.4,20899.4,5955.9,55838.0,55838.0,5349.0,59.0,0.0,11.0,5.0
3,25.740676,-80.259778,13.367660,5000.0,1837.0,5234.0,3152.2,8549.3,4062.0,25623.7,3199.4,15719.2,19.0,0.0,5.0,5.0
4,25.963201,-80.182147,12.834681,5713.0,2140.0,16500.0,3373.8,20809.2,4580.4,58837.2,58837.2,3581.9,38.0,0.0,7.0,5.0


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,avno60plus,category,0.0,0.0,2.0,"0.0, 1.0"
1,LATITUDE,float64,0.0,0.0,13776.0,"25.9562, 25.9535, 25.5322, 25.9716, 25.7407, 25.9632, 25.6189, 25.6958, 25.931, 25.7023"
2,LONGITUDE,float64,0.0,0.0,13776.0,"-80.1787, -80.3431, -80.3798, -80.182, -80.2598, -80.1821, -80.3523, -80.4106, -80.2799, -80.408"
3,SALE_PRC,float64,0.0,0.0,2106.0,"12.4292, 12.6115, 12.4684, 12.5062, 12.5776, 12.5425, 12.7657, 12.4875, 12.5602, 12.2549"
4,LND_SQFOOT,float64,0.0,0.0,4696.0,"7500.0, 5000.0, 6000.0, 7875.0, 8250.0, 8000.0, 15000.0, 5500.0, 5250.0, 10000.0"
5,TOT_LVG_AREA,float64,0.0,0.0,2978.0,"3079.0, 3199.0, 2176.0, 1701.0, 1440.0, 2578.0, 2193.0, 2091.0, 1699.0, 2552.0"
6,SPEC_FEAT_VAL,float64,0.0,0.0,7583.0,"0.0, 550.0, 440.0, 4800.0, 1200.0, 3200.0, 2240.0, 2460.0, 2200.0, 1296.0"
7,RAIL_DIST,float64,0.0,0.0,13235.0,"50.0, 49.9, 16135.4, 1675.8, 14690.8, 7970.2, 7529.5, 2539.6, 77.9, 1285.8"
8,OCEAN_DIST,float64,0.0,0.0,13617.0,"28968.2, 47914.1, 26663.1, 36924.2, 9893.2, 19051.5, 17283.2, 54708.0, 24602.5, 24961.0"
9,WATER_DIST,float64,0.0,0.0,13218.0,"0.0, 7.2, 3644.1, 2750.0, 2889.2, 6252.7, 11.0, 523.4, 10.5, 10267.8"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
LATITUDE,13776.0,25.728260,0.140631,25.434333,25.974382
LONGITUDE,13776.0,-80.327907,0.089077,-80.542172,-80.119746
SALE_PRC,13776.0,12.712804,0.568216,11.184421,14.790070
LND_SQFOOT,13776.0,8623.955285,6063.466105,1248.000000,57064.000000
TOT_LVG_AREA,13776.0,2061.495499,814.355733,854.000000,6287.000000
SPEC_FEAT_VAL,13776.0,9604.750871,13923.548933,0.000000,175020.000000
RAIL_DIST,13776.0,8354.846762,6175.566582,10.500000,29621.500000
OCEAN_DIST,13776.0,31710.960714,17609.827844,236.100000,75744.900000
WATER_DIST,13776.0,11994.245274,11941.117742,0.000000,50399.800000
CNTR_DIST,13776.0,68625.835402,31990.844022,3825.600000,159976.500000


In [7]:
# Categorical Feature Statistics
cat_stats

value  count   pct
column     rank                   
avno60plus 1      0.0  13570  98.5
           2      1.0    206   1.5

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,0.736,0.564,0.323,0.002,log,240862.3,930662.5,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to miami_housing/019d5cdf-824f-7703-84b4-8e6949a71166
019d5cdf-824f-7703-84b4-8e6949a71166
847eafc1699361a23846d8aa5d3ec1abb37ff0179a81111863e67ec2921a6b68
